# Agentic Retrieval-Augmented Generation (RAG) System

This notebook builds a document question-answering system using a RAG architecture.

Pipeline:

PDF → Chunking → Embeddings → Vector Database → Semantic Retrieval → LLM Answer

Technologies used:

- LangChain
- Sentence Transformers (all-MiniLM-L6-v2)
- Qdrant Vector Database
- Llama 3.1 via Groq
- Gradio Web Interface

# Imports

In [1]:
import os
import numpy as np
from pypdf import PdfReader

from dotenv import load_dotenv

from langchain.schema import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain.embeddings import HuggingFaceEmbeddings

from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

from langchain.tools import tool
from langchain.agents import create_react_agent, AgentExecutor
from langchain import hub

from langchain_groq import ChatGroq

import gradio as gr


# load environment variables
load_dotenv()

# read API key
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

print("API Loaded:", GROQ_API_KEY is not None)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

API Loaded: True


# Extracting Text from PDFs

In [2]:
from langchain.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
   model_name="BAAI/bge-base-en-v1.5",
)

# embeddings = embedding_model.embed_documents(
#     [chunk.page_content for chunk in chunks]
# )

# print("Number of embeddings:", len(embeddings))
# print("Embedding vector size:", len(embeddings[0]))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
from qdrant_client import QdrantClient

client = QdrantClient(path="./qdrant_storage")
collection_name = "rag_collection"

In [4]:
if client.collection_exists(collection_name):
    client.delete_collection(collection_name)

client.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(
        size=768,
        distance=Distance.COSINE
    )
)

True

In [5]:
# points = []

# for i, embedding in enumerate(embeddings):
#     points.append(
#         PointStruct(
#             id=i,
#             vector=embedding,
#             payload={
#                 "text": chunks[i].page_content
#             }
#         )
#     )

# client.upsert(
#     collection_name=collection_name,
#     points=points
# )

# print("Embeddings stored!")

NameError: name 'embeddings' is not defined

In [ ]:
@tool
def search_resume(question: str):
    """
    Search the resume and return relevant sections.

    Use this tool whenever a question asks about:
    - projects
    - skills
    - education
    - experience
    - achievements
    """

    query_embedding = embedding_model.embed_query(question)

    results = client.query_points(
        collection_name=collection_name,
        query=query_embedding,
        limit=8
    )

    # print("\nRetrieved Chunks:\n")

    # for r in results.points:
    #     print(r.payload["text"])
    #     print("\n----------------\n")

    context = "\n".join([r.payload["text"] for r in results.points[:3]])

    return context

In [ ]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key= GROQ_API_KEY,
    temperature=0
)

In [ ]:
from langchain.prompts import PromptTemplate

react_prompt = PromptTemplate.from_template("""
You are an intelligent assistant that analyzes resumes.

You have access to the following tools:

{tools}

When answering a question:
1. Always use the search_resume tool first.
2. Read the retrieved context carefully.
3. Answer the question using only the retrieved information.

Use the following format:

Question: the input question
Thought: you should think about what to do
Action: the tool to use, should be one of [{tool_names}]
Action Input: the input to the tool
Observation: the result of the tool
... (this Thought/Action/Observation can repeat)
Thought: I now know the final answer
Final Answer: the answer to the question

Question: {input}

{agent_scratchpad}
""")

In [ ]:
tools = [search_resume]

agent = create_react_agent(
    llm,
    tools,
    react_prompt
)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)

In [ ]:
def ask_question(question):
    try:
        response = agent_executor.invoke(
            {"input": question}
        )

        return response.get("output", str(response))

    except Exception as e:
        return str(e)

In [ ]:
def ask_llm(context, question):
    prompt = f"""
You are an AI assistant that answers questions based only on the provided context.

Context:
{context}

Question:
{question}

Answer clearly and concisely using only the context.
"""

    response = llm.invoke(prompt)

    return response.content

In [ ]:
def rag_query(question):

    query_embedding = embedding_model.embed_query(question)

    results = client.query_points(
        collection_name=collection_name,
        query=query_embedding,
        limit=5
    )

    # print("\nRetrieved Chunks:\n")

    # for r in results.points:
    #     print(r.payload["text"])
    #     print("\n-----------------\n")

    context = "\n".join([r.payload["text"] for r in results.points])

    answer = ask_llm(context, question)

    return answer

In [ ]:
# print(rag_query("What is this document about?"))

In [ ]:
import gradio as gr

# Global state to track if a PDF has been indexed
indexed = False

def index_pdf(pdf_file):
    global indexed, client, embedding_model

    # Read the uploaded PDF
    reader = PdfReader(pdf_file)
    text = ""
    for page in reader.pages:
        text += page.extract_text()

    # Chunk it
    documents = [Document(page_content=text)]
    chunks = text_splitter.split_documents(documents)

    # Embed it
    embeddings = embedding_model.embed_documents(
        [chunk.page_content for chunk in chunks]
    )

    # Reset and re-index Qdrant
    if client.collection_exists(collection_name):
        client.delete_collection(collection_name)

    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=768, distance=Distance.COSINE)
    )

    points = [
        PointStruct(
            id=i,
            vector=emb,
            payload={"text": chunks[i].page_content}
        )
        for i, emb in enumerate(embeddings)
    ]
    client.upsert(collection_name=collection_name, points=points)
    indexed = True

    return "✅ PDF indexed! You can now ask questions below."


def chat(message, history):
    if not indexed:
        return "⚠️ Please upload a PDF first."
    return ask_question(message)


with gr.Blocks(title="Agentic RAG with LLaMA 3.1") as demo:
    gr.Markdown("# 🤖 Agentic RAG with LLaMA 3.1")
    gr.Markdown("Upload any PDF, then ask questions about it.")

    with gr.Row():
        pdf_input = gr.File(label="📄 Upload PDF", file_types=[".pdf"])
        status = gr.Textbox(label="Status", interactive=False)

    pdf_input.change(fn=index_pdf, inputs=pdf_input, outputs=status)

    gr.ChatInterface(
        fn=chat,
        examples=[
            "What is this document about?",
            "Summarize the key points.",
            "What are the main skills mentioned?"
        ]
    )

demo.launch()